<a href="https://colab.research.google.com/github/garvagrawalhere/web-crawler-and-mini-search-engine/blob/main/Web_crawler_and_Mini_Search_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Web crawler and Mini Search Engine
###By Garv Agrawal

In [1]:
!pip -q install httpx beautifulsoup4 fastapi uvicorn scikit-learn nest-asyncio

In [2]:
import asyncio
import time
import re
import string
from collections import deque
from urllib.parse import urljoin, urlsplit, urlunsplit

import httpx
from bs4 import BeautifulSoup

import nest_asyncio
nest_asyncio.apply()

##Adding the Seed URL and configuring settings for crawler





In [3]:
# PROJECT CONFIGURATION


SEED_URL = input(
    "Enter the seed URL to crawl: "
).strip()

MAX_PAGES = int(input(
    "Maximum number of pages to crawl [default 50]: "
) or 50)

MAX_DEPTH = int(input(
    "Maximum crawl depth [default 2]: "
) or 2)

REQUEST_TIMEOUT = 5

# Extract domain from seed URL
parsed_seed = urlsplit(SEED_URL)

ALLOWED_DOMAIN = parsed_seed.netloc

print("\n==============================")
print("Crawler Configuration")
print("==============================")
print("Seed URL     :", SEED_URL)
print("Domain       :", ALLOWED_DOMAIN)
print("Max pages    :", MAX_PAGES)
print("Max depth    :", MAX_DEPTH)

Enter the seed URL to crawl: https://www.geeksforgeeks.org/python/python-programming-language-tutorial/
Maximum number of pages to crawl [default 50]: 20
Maximum crawl depth [default 2]: 2

Crawler Configuration
Seed URL     : https://www.geeksforgeeks.org/python/python-programming-language-tutorial/
Domain       : www.geeksforgeeks.org
Max pages    : 20
Max depth    : 2


##URL Normalization

In [4]:
def normalize_url(url):
    parts = urlsplit(url)

    return urlunsplit((
        parts.scheme.lower(),
        parts.netloc.lower(),
        parts.path or "/",
        parts.query,
        ""
    ))

Can Test the normalizer via this cell in comment:

In [5]:
# test_url = "https://example.com/about#team"

# print(normalize_url(test_url))

##Extracting links from the page

In [6]:
def extract_links(html, current_url):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    links = set()

    for tag in soup.find_all("a"):

        href = tag.get("href")

        if not href:
            continue

        # Convert relative URL → absolute URL
        absolute_url = urljoin(
            current_url,
            href
        )

        # Normalize
        normalized_url = normalize_url(
            absolute_url
        )

        links.add(normalized_url)

    return links

In [7]:
async def fetch_page(client, url):

    try:

        response = await client.get(
            url,
            timeout=REQUEST_TIMEOUT,
            follow_redirects=True
        )

        if response.status_code == 200:
            return response.text

        print(
            f"⚠️ Skipping {url} "
            f"(HTTP {response.status_code})"
        )

        return None

    except httpx.RequestError as e:

        print(
            f"⚠️ Request failed: {url}"
        )

        return None

##Crawler- Using BFS

In [8]:
async def crawl():

    queue = deque()

    queue.append(
        (
            normalize_url(SEED_URL),
            0
        )
    )

    visited = set()

    pages = {}

    async with httpx.AsyncClient(
        headers={
            "User-Agent": "MiniSearchCrawler/1.0"
        }
    ) as client:

        while (
            queue
            and len(visited) < MAX_PAGES
        ):

            url, depth = queue.popleft()

            # Already crawled?
            if url in visited:
                continue

            # Maximum depth reached?
            if depth > MAX_DEPTH:
                continue

            # Stay inside selected domain
            if urlsplit(url).netloc != ALLOWED_DOMAIN:
                continue

            visited.add(url)

            print(
                f"[{len(visited)}/{MAX_PAGES}] "
                f"Crawling: {url}"
            )

            html = await fetch_page(
                client,
                url
            )

            if html is None:
                continue

            # Store webpage
            pages[url] = html

            # Don't discover more links
            # from maximum-depth pages
            if depth == MAX_DEPTH:
                continue

            links = extract_links(
                html,
                url
            )

            for link in links:

                if link not in visited:

                    queue.append(
                        (
                            link,
                            depth + 1
                        )
                    )

    return pages

##Applying the functions to start the crawling

In [9]:
start_time = time.perf_counter()

pages = await crawl()

crawl_time = (
    time.perf_counter()
    - start_time
)


print("CRAWL COMPLETE")


print(
    "Pages crawled:",
    len(pages)
)

print(
    f"Crawl time: {crawl_time:.2f} seconds"
)

[1/20] Crawling: https://www.geeksforgeeks.org/python/python-programming-language-tutorial/
[2/20] Crawling: https://www.geeksforgeeks.org/python/sets-in-python/
[3/20] Crawling: https://www.geeksforgeeks.org/campus-training-program/
[4/20] Crawling: https://www.geeksforgeeks.org/data-science/statsmodel-library-tutorial/
[5/20] Crawling: https://www.geeksforgeeks.org/dsa/dsa-tutorial-learn-data-structures-and-algorithms/
[6/20] Crawling: https://www.geeksforgeeks.org/python/python-mongodb-tutorial/
[7/20] Crawling: https://www.geeksforgeeks.org/python/introduction-to-python/
[8/20] Crawling: https://www.geeksforgeeks.org/quizzes/functions-python-gq/
[9/20] Crawling: https://www.geeksforgeeks.org/python/python-language-advantages-disadvantages-applications/
[10/20] Crawling: https://www.geeksforgeeks.org/legal/
[11/20] Crawling: https://www.geeksforgeeks.org/python/python-classes-and-objects/
[12/20] Crawling: https://www.geeksforgeeks.org/nation-skill-up/
[13/20] Crawling: https://www.



---

##Crawler functionality finished. Now we move on to preprocessing of the text generated by crawler




###Configuring preprocessor

In [10]:
STOP_WORDS = {
    "the", "is", "a", "an",
    "and", "or", "of", "to",
    "in", "on", "for", "with",
    "this", "that", "are",
    "was", "were", "be",
    "as", "at", "by"
}


def preprocess_text(html):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    # Remove elements that aren't useful
    # for our search index
    for tag in soup([
        "script",
        "style",
        "noscript"
    ]):
        tag.decompose()

    text = soup.get_text(
        separator=" "
    )

    # Lowercase
    text = text.lower()

    # Remove punctuation
    text = text.translate(
        str.maketrans(
            "",
            "",
            string.punctuation
        )
    )

    # Tokenize
    tokens = text.split()

    # Remove stop words
    tokens = [
        word
        for word in tokens
        if word not in STOP_WORDS
    ]

    return tokens

###Processing crawled pages

In [11]:
# ==============================
# TEXT PREPROCESSING
# ==============================

documents = {}

for url, html in pages.items():

    tokens = preprocess_text(html)

    if tokens:
        documents[url] = tokens


print("Documents indexed:", len(documents))

print("\n" + "=" * 80)
print("Processed documents and their respective tokens")
print("=" * 80)

for i, (url, tokens) in enumerate(documents.items(), start=1):

    print(f"\nDOCUMENT {i}")
    print("-" * 80)

    print("URL:")
    print(url)

    print("\nTokens:")
    print(tokens)

Documents indexed: 20

Processed documents and their respective tokens

DOCUMENT 1
--------------------------------------------------------------------------------
URL:
https://www.geeksforgeeks.org/python/python-programming-language-tutorial/

Tokens:
['python', 'tutorial', 'geeksforgeeks', 'courses', 'tutorials', 'interview', 'prep', 'python', 'tutorial', 'data', 'types', 'interview', 'questions', 'examples', 'quizzes', 'dsa', 'python', 'data', 'science', 'numpy', 'pandas', 'practice', 'django', 'flask', 'python', 'tutorial', 'last', 'updated', '3', 'aug', '2026', 'python', 'one', 'most', 'popular', 'programming', 'languages', 'it’s', 'simple', 'use', 'packed', 'features', 'supported', 'wide', 'range', 'libraries', 'frameworks', 'its', 'clean', 'syntax', 'makes', 'it', 'beginnerfriendly', 'highlevel', 'language', 'used', 'data', 'science', 'automation', 'ai', 'web', 'development', 'more', 'known', 'its', 'readability', 'which', 'means', 'code', 'easier', 'write', 'understand', 'maint

###Ok, got the tokens. now we make an Inverted Index

In [12]:
inverted_index = {}

for url, tokens in documents.items():

    for token in tokens:

        if token not in inverted_index:

            inverted_index[token] = set()

        inverted_index[token].add(url)


print(
    "Unique terms:",
    len(inverted_index)
)


Unique terms: 2835


##Calculating TF-IDF weights for these unique terms

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
urls = list(documents.keys())

corpus = [
    " ".join(documents[url])
    for url in urls
]

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    corpus
)

print(
    "TF-IDF matrix shape:",
    tfidf_matrix.shape
)

TF-IDF matrix shape: (20, 2780)


###Search Engine

In [14]:
import numpy as np


def search_engine(query, top_k=5):

    query_vector = vectorizer.transform(
        [query]
    )

    scores = (
        tfidf_matrix
        @ query_vector.T
    ).toarray().flatten()

    ranked_indices = np.argsort(
        scores
    )[::-1]

    results = []

    for index in ranked_indices:

        score = scores[index]

        if score <= 0:
            continue

        results.append({
            "url": urls[index],
            "score": float(score)
        })

        if len(results) >= top_k:
            break

    return results

In [15]:
query = input(
    "Enter the term/terms to search: "
).strip()

results = search_engine(
    query
)

print("\n==============================")
print("SEARCH RESULTS")
print("==============================")

if not results:

    print("No matching pages found.")

else:

    for rank, result in enumerate(
        results,
        start=1
    ):

        print(
            f"\n{rank}. "
            f"{result['url']}"
        )

        print(
            f"   TF-IDF Score: "
            f"{result['score']:.4f}"
        )

Enter the term/terms to search: Variable

SEARCH RESULTS

1. https://www.geeksforgeeks.org/quizzes/functions-python-gq/
   TF-IDF Score: 0.1581

2. https://www.geeksforgeeks.org/python/python-keywords/
   TF-IDF Score: 0.0945

3. https://www.geeksforgeeks.org/quizzes/data-type-gq/
   TF-IDF Score: 0.0580

4. https://www.geeksforgeeks.org/quizzes/python-control-flow-conditional-logic-quiz/
   TF-IDF Score: 0.0277

5. https://www.geeksforgeeks.org/quizzes/python-oops-quiz/
   TF-IDF Score: 0.0227


Now making a Fastapi search app

In [16]:
from fastapi import FastAPI, Query

app = FastAPI(
    title="Mini Search Engine API",
    description="Domain-specific search engine using TF-IDF"
)


@app.get("/search")
def search(
    q: str = Query(
        ...,
        description="Search query"
    )
):

    results = search_engine(q)

    return {
        "query": q,
        "results": results
    }


print("FastAPI application created successfully.")

FastAPI application created successfully.


In [17]:
from fastapi.testclient import TestClient

client = TestClient(app)

query = input(
    "Enter a search query to test the API: "
).strip()

response = client.get(
    "/search",
    params={"q": query}
)

print("\nStatus code:", response.status_code)

print("\nAPI response:")

print(response.json())

Enter a search query to test the API: what is python

Status code: 200

API response:
{'query': 'what is python', 'results': [{'url': 'https://www.geeksforgeeks.org/quizzes/python-control-flow-conditional-logic-quiz/', 'score': 0.29576933509058845}, {'url': 'https://www.geeksforgeeks.org/quizzes/python-oops-quiz/', 'score': 0.22272514487267692}, {'url': 'https://www.geeksforgeeks.org/quizzes/functions-python-gq/', 'score': 0.17667572809757232}, {'url': 'https://www.geeksforgeeks.org/quizzes/data-type-gq/', 'score': 0.17468656390229997}, {'url': 'https://www.geeksforgeeks.org/python/python-language-advantages-disadvantages-applications/', 'score': 0.13399584724876498}]}


##Search engine made.



---




##Now comparing sequential fetching to Concurrent fetching

###Sequential fetching

In [18]:
async def fetch_sequential(urls):

    results = {}

    async with httpx.AsyncClient(
        headers={
            "User-Agent": "MiniSearchCrawler/1.0"
        }
    ) as client:

        for url in urls:

            html = await fetch_page(
                client,
                url
            )

            if html is not None:
                results[url] = html

    return results

###Concurrent fetching

In [19]:
CONCURRENCY_LIMIT = 10


async def fetch_concurrent(urls):

    semaphore = asyncio.Semaphore(
        CONCURRENCY_LIMIT
    )

    async with httpx.AsyncClient(
        headers={
            "User-Agent": "MiniSearchCrawler/1.0"
        }
    ) as client:

        async def fetch_with_limit(url):

            async with semaphore:

                html = await fetch_page(
                    client,
                    url
                )

                return url, html

        tasks = [
            fetch_with_limit(url)
            for url in urls
        ]

        results = await asyncio.gather(
            *tasks
        )

    return {
        url: html
        for url, html in results
        if html is not None
    }

In [20]:
benchmark_urls = list(pages.keys())

print(
    "URLs used for benchmark:",
    len(benchmark_urls)
)


# ==============================
# SEQUENTIAL
# ==============================

start = time.perf_counter()

sequential_pages = await fetch_sequential(
    benchmark_urls
)

sequential_time = (
    time.perf_counter() - start
)


# ==============================
# CONCURRENT
# ==============================

start = time.perf_counter()

concurrent_pages = await fetch_concurrent(
    benchmark_urls
)

concurrent_time = (
    time.perf_counter() - start
)


# ==============================
# RESULTS
# ==============================

print("\n==============================")
print("CRAWLING PERFORMANCE")
print("==============================")

print(
    f"Sequential time : "
    f"{sequential_time:.2f} seconds"
)

print(
    f"Concurrent time : "
    f"{concurrent_time:.2f} seconds"
)

if sequential_time > 0:

    improvement = (
        (sequential_time - concurrent_time)
        / sequential_time
    ) * 100

    speedup = (
        sequential_time
        / concurrent_time
    )

    print(
        f"Time reduction  : "
        f"{improvement:.2f}%"
    )

    print(
        f"Speedup         : "
        f"{speedup:.2f}x"
    )

URLs used for benchmark: 20

CRAWLING PERFORMANCE
Sequential time : 0.54 seconds
Concurrent time : 0.21 seconds
Time reduction  : 60.65%
Speedup         : 2.54x


##Throughput

In [21]:
sequential_throughput = (
    len(benchmark_urls)
    / sequential_time
)

concurrent_throughput = (
    len(benchmark_urls)
    / concurrent_time
)

print("\n==============================")
print("THROUGHPUT")
print("==============================")

print(
    f"Sequential: "
    f"{sequential_throughput:.2f} pages/sec"
)

print(
    f"Concurrent: "
    f"{concurrent_throughput:.2f} pages/sec"
)


THROUGHPUT
Sequential: 36.89 pages/sec
Concurrent: 93.73 pages/sec


##Latency

In [22]:
queries_input = input(
    "Enter search queries separated by commas: "
).strip()

queries = [
    q.strip()
    for q in queries_input.split(",")
    if q.strip()
]

latencies = []

for query in queries:

    start = time.perf_counter()

    results = search_engine(query)

    latency = (
        time.perf_counter() - start
    ) * 1000

    latencies.append(latency)

    print(
        f"{query!r} → "
        f"{latency:.2f} ms"
    )


if latencies:

    average_latency = (
        sum(latencies)
        / len(latencies)
    )

    sorted_latencies = sorted(latencies)

    p95_index = int(
        0.95 * len(sorted_latencies)
    ) - 1

    p95_latency = sorted_latencies[
        max(0, p95_index)
    ]

    print("\n==============================")
    print("SEARCH PERFORMANCE")
    print("==============================")

    print(
        f"Average latency: "
        f"{average_latency:.2f} ms"
    )

    print(
        f"P95 latency: "
        f"{p95_latency:.2f} ms"
    )

Enter search queries separated by commas: python, function, syntax
'python' → 1.53 ms
'function' → 1.60 ms
'syntax' → 0.98 ms

SEARCH PERFORMANCE
Average latency: 1.37 ms
P95 latency: 1.53 ms


In [23]:
print("\n")
print("=" * 60)
print("FINAL PROJECT METRICS")
# print("=" * 60)

print(
    f"Pages crawled          : {len(pages)}"
)

print(
    f"Documents indexed      : {len(documents)}"
)

print(
    f"Initial crawl time     : "
    f"{crawl_time:.2f} sec"
)

print(
    f"Sequential fetch time  : "
    f"{sequential_time:.2f} sec"
)

print(
    f"Concurrent fetch time  : "
    f"{concurrent_time:.2f} sec"
)

print(
    f"Latency reduction      : "
    f"{((sequential_time-concurrent_time)*100)/sequential_time:.2f} %"
)

print(
    f"Sequential throughput  : "
    f"{sequential_throughput:.2f} pages/sec"
)

print(
    f"Concurrent throughput  : "
    f"{concurrent_throughput:.2f} pages/sec"
)

print(
    f"Search avg latency     : "
    f"{average_latency:.2f} ms"
)

print(
    f"Search P95 latency     : "
    f"{p95_latency:.2f} ms"
)

print("=" * 60)



FINAL PROJECT METRICS
Pages crawled          : 20
Documents indexed      : 20
Initial crawl time     : 2.38 sec
Sequential fetch time  : 0.54 sec
Concurrent fetch time  : 0.21 sec
Latency reduction      : 60.65 %
Sequential throughput  : 36.89 pages/sec
Concurrent throughput  : 93.73 pages/sec
Search avg latency     : 1.37 ms
Search P95 latency     : 1.53 ms
